# MLX tutorial for interpretability work

Companion to `hooked_transformer_tutorial.ipynb`, same four capabilities, this time in **MLX** (Apple's array framework, native to Apple Silicon):

1. Load a small model
2. A basic forward pass
3. Grabbing activations
4. Intervening on activations (causal ablation)
5. Gradients: parameters, and activations via a first-class VJP

MLX's programming model differs from PyTorch/TransformerLens in ways that change how each of these is done:

- **No device management.** MLX arrays live in unified memory and ops run on the GPU by default on Apple Silicon — there's no `.to(device)` / MPS correctness caveat to think about.
- **No hook-point system.** MLX modules are plain Python objects with a `__call__` you define yourself. There's nothing like TransformerLens's `HookPoint`/`run_with_hooks` — and a tempting shortcut, monkeypatching an *instance's* `__call__`, silently does nothing (Python resolves `obj(x)` via `type(obj).__call__`, not the instance's `__dict__`). The idiomatic way to get an intermediate activation is simply to call the model's own submodules yourself in a loop, instead of hooking.
- **Functional autodiff.** No `.backward()`, no `.grad` sitting on parameters. You call `mx.grad`/`nn.value_and_grad` on a pure function and get gradients back as a matching pytree (nested dict).
- **VJPs are first-class.** `mx.vjp(fn, primals, cotangents)` directly gives what we had to build by hand with `bwd_hooks` in the PyTorch notebook.
- **Arrays are lazy.** An op builds a graph node, not a value; call `mx.eval(...)` (or `.item()`, which evaluates implicitly) to actually run it.

Model used: **SmolLM2-135M-Instruct** (`mlx-community/SmolLM2-135M-Instruct`) — a 135M-param Llama-architecture model (30 layers, d_model 576), pre-converted to MLX, small enough for fast iteration on an M2.

Einops is a library for simple, readable, self documenting code for tedious matrix multiplications, simple rearrangements, flattening, etc which are notoriously difficult to read in torch and numpy code. Einops supports mlx while offering the exact same interface for a bunch of backends(torch, numpy, etc) which makes it very convenient to use.

In [32]:
import mlx.core as mx
import mlx.nn as nn
from einops import rearrange
from mlx_lm import load
from mlx_lm.models.base import create_attention_mask, scaled_dot_product_attention

mx.random.seed(0)

## 1. Load the model

`mlx_lm.load` downloads the (already MLX-converted) weights and returns `(model, tokenizer)`. Printing the model shows its module tree directly — this *is* the model's source-level structure, so we know exactly what to call by hand later.

In [3]:
model, tokenizer = load("mlx-community/SmolLM2-135M-Instruct")

print(type(model))
print("n layers:", len(model.model.layers))
print("hidden size:", model.args.hidden_size)
print("tie_word_embeddings:", model.args.tie_word_embeddings)  # no separate lm_head if True

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

<class 'mlx_lm.models.llama.Model'>
n layers: 30
hidden size: 576
tie_word_embeddings: True


## 2. Basic forward pass

`tokenizer.encode`/`.decode` handle tokenization. Calling the model returns logits `[batch, seq, vocab]`, same shape convention as TransformerLens.

In [4]:
prompt = "The quick brown fox jumps over the lazy"
ids = tokenizer.encode(prompt)
inputs = mx.array(ids)[None, :]
print("tokens shape:", inputs.shape)
print("tokens:", [tokenizer.decode([t]) for t in ids])

logits = model(inputs)
print("logits shape:", logits.shape)

next_token = logits[0, -1].argmax()
print("predicted next token:", repr(tokenizer.decode([next_token.item()])))

tokens shape: (1, 8)
tokens: ['The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy']
logits shape: (1, 8, 49152)
predicted next token: ' dog'


## 3. Grabbing activations

No hooks needed — `Model.__call__`/`LlamaModel.__call__` (see `mlx_lm/models/llama.py`) is just:

```python
h = self.embed_tokens(inputs)
for layer in self.layers:
    h = layer(h, mask, cache)
return self.norm(h)
```

So we get the residual stream after every layer by running that same loop ourselves and stashing `h`. This is verified below to produce bit-identical logits to `model(inputs)`.

In [5]:
h = model.model.embed_tokens(inputs)
mask = create_attention_mask(h, None)

resid_after_layer = []
for layer in model.model.layers:
    h = layer(h, mask, cache=None)
    resid_after_layer.append(h)

final = model.model.norm(h)
manual_logits = model.model.embed_tokens.as_linear(final)  # tied embedding == lm_head

print("resid after layer 0:", resid_after_layer[0].shape)   # [batch, seq, d_model]
print("resid after layer 10:", resid_after_layer[10].shape)
print("max abs diff vs model(inputs):", mx.abs(manual_logits - logits).max().item())

resid after layer 0: (1, 8, 576)
resid after layer 10: (1, 8, 576)
max abs diff vs model(inputs): 0.0


## 4. Intervening: single-head ablation

To ablate one attention head we need to get inside `Attention.__call__` (it combines all heads through `o_proj`), so for the target layer we re-run its attention math by hand — same q/k/v/rope/attention calls the layer itself uses — zero one head's output, then continue normally through the rest of the model.

In [6]:
layer_to_ablate = 3
head_to_ablate = 2

def ablated_attn(attn, x, mask):
    query, key, value = attn.q_proj(x), attn.k_proj(x), attn.v_proj(x)
    query = rearrange(
        query, "batch seq (query_heads head_dim) -> batch query_heads seq head_dim",
        query_heads=attn.n_heads,
    )
    key = rearrange(
        key, "batch seq (key_value_heads head_dim) -> batch key_value_heads seq head_dim",
        key_value_heads=attn.n_kv_heads,
    )
    value = rearrange(
        value, "batch seq (key_value_heads head_dim) -> batch key_value_heads seq head_dim",
        key_value_heads=attn.n_kv_heads,
    )
    query, key = attn.rope(query), attn.rope(key)
    attn_out = scaled_dot_product_attention(
        query, key, value, cache=None, scale=attn.scale, mask=mask
    )
    attn_out = rearrange(
        attn_out, "batch query_heads seq head_dim -> batch seq query_heads head_dim"
    )
    attn_out[:, :, head_to_ablate, :] = mx.zeros_like(attn_out[:, :, head_to_ablate, :])
    attn_out = rearrange(
        attn_out, "batch seq query_heads head_dim -> batch seq (query_heads head_dim)"
    )
    return attn.o_proj(attn_out)

h = model.model.embed_tokens(inputs)
mask = create_attention_mask(h, None)
for i, layer in enumerate(model.model.layers):
    if i == layer_to_ablate:
        r = ablated_attn(layer.self_attn, layer.input_layernorm(h), mask)
        h = h + r
        r = layer.mlp(layer.post_attention_layernorm(h))
        h = h + r
    else:
        h = layer(h, mask, cache=None)
ablated_logits = model.model.embed_tokens.as_linear(model.model.norm(h))

orig_probs = mx.softmax(logits[0, -1], axis=-1)
ablated_probs = mx.softmax(ablated_logits[0, -1], axis=-1)

print("original top token:", repr(tokenizer.decode([orig_probs.argmax().item()])))
print("ablated top token:", repr(tokenizer.decode([ablated_probs.argmax().item()])))
print("max prob change:", mx.abs(orig_probs - ablated_probs).max().item())

original top token: ' dog'
ablated top token: ' dog'
max prob change: 0.01171875


## 5. Gradients

### 5a. Parameter gradients

No `.backward()`. `nn.value_and_grad(model, loss_fn)` wraps `mx.value_and_grad` to differentiate w.r.t. `model.trainable_parameters()`, returning a nested-dict pytree of gradients mirroring the model's structure.

In [7]:
def loss_fn(model, inputs):
    logits = model(inputs[:, :-1])
    targets = inputs[:, 1:]
    return nn.losses.cross_entropy(logits, targets).mean()

loss_and_grad_fn = nn.value_and_grad(model, loss_fn)
loss, grads = loss_and_grad_fn(model, inputs)

print("loss:", loss.item())

q_grad = grads["model"]["layers"][0]["self_attn"]["q_proj"]["weight"]
print("layer 0 q_proj grad shape:", q_grad.shape)
print("layer 0 q_proj grad norm:", mx.sqrt((q_grad ** 2).sum()).item())

loss: 2.578125
layer 0 q_proj grad shape: (576, 576)
layer 0 q_proj grad norm: 0.1640625


### 5b. Activation gradients: `mx.vjp`

In the PyTorch notebook, getting the gradient of a later activation w.r.t. an earlier one to get the Jacobian which we need for the J-Lens probably requires writing the forward pass by hand and using pytorch's vjp function whatever it may be called. This is tedious, I'm pretty sure there's no way to get around this using stuff from `TransformerLens`. In MLX what I need to do is the same however, the forward pass is just a loop of `h = layer(h, mask)` where the mask comes from `create_attention_mask()` which is super easy to use.

In [10]:
layer_i, layer_j = 3, 10

h = model.model.embed_tokens(inputs)
mask = create_attention_mask(h, None)
for layer in model.model.layers[: layer_i + 1]:
    h = layer(h, mask, cache=None)
h_i = h  # residual stream after layer_i

def f(h_i):
    h = h_i
    for layer in model.model.layers[layer_i + 1 : layer_j + 1]:
        h = layer(h, mask, cache=None)
    return h

h_j = f(h_i)
# Kind of forgot what the shapes of h_i and h_j are, so let's print them out
print("h_i shape:", h_i.shape, " h_j shape:", h_j.shape)

h_i shape: (1, 8, 576)  h_j shape: (1, 8, 576)


In [23]:
# so it is batch x seq x d_model, and we want the Jacobian of the last position in the sequence, so we can contract over the batch and seq dimensions with an all-ones cotangent vector
cotangent = mx.zeros_like(h_j)
cotangent[:, -1, :] = 1.0  # cotangent = 1 across every feature at the last position

(h_j_out,), (grad_h_i,) = mx.vjp(f, (h_i,), (cotangent,))


# A single VJP call always returns something shaped like h_i, not a
# Jacobian: it's d(sum_features h_j[0, -1, :]) / d(h_i), one gradient vector,
# contracted over h_j's feature dim by the all-ones cotangent -- a weighted SUM
# of Jacobian rows, not the Jacobian itself. The next cell builds the actual
# [d_model_j, d_model_i] Jacobian matrix.
print("h_i shape:", h_i.shape, " h_j shape:", h_j.shape)
print("single VJP result shape (== h_i's shape, NOT a Jacobian):", grad_h_i.shape)
print("grad norm:", mx.sqrt((grad_h_i ** 2).sum()).item())

h_i shape: (1, 8, 576)  h_j shape: (1, 8, 576)
single VJP result shape (== h_i's shape, NOT a Jacobian): (1, 8, 576)
grad norm: 37.75


### The actual Jacobian, via `mx.vmap`

A full Jacobian needs one VJP per output dimension (one-hot cotangents), but MLX can batch all of them into a single call with `mx.vmap`, instead of a Python loop. `mx.vmap(vjp_row)` runs `vjp_row` once per row of the `cotangents` batch and stacks the results, just expressed as a function transform instead of manual array-replication plumbing.

In [25]:
pos = -1  # compute the Jacobian at a single sequence position (keeps it a 2D matrix)
d_model_j = h_j.shape[-1]

# One-hot cotangents, one per output feature, all at the same source/target position.
basis = mx.eye(d_model_j)
cotangents = mx.zeros((d_model_j,) + h_j.shape)
cotangents[:, 0, pos, :] = basis
# compute for all positions in the sequence, not just pos

# The following einops code doesn't work as expected specifically for mlx, check out bug with einops later
# tmp = rearrange(basis, "d_model_i d_model_j -> d_model_i 1 1 d_model_j")
# cotangents = repeat(tmp, "d_model_i 1 1 d_model_j -> d_model_i 1 seq_len d_model_j", seq_len=h_j.shape[1])

def vjp_row(cotangent):
    _, (grad,) = mx.vjp(f, (h_i,), (cotangent,))
    # return grad[:, pos, :]  # [1, d_model_i]
    return grad

jac_rows = mx.vmap(vjp_row)(cotangents)  # [d_model_j, 1, d_model_i]
jacobian = jac_rows[:, 0, -1, :]  # [d_model_j, d_model_i]

print("jacobian shape:", jacobian.shape)  # d(h_j[pos]) / d(h_i[pos])
print("jacobian norm:", mx.sqrt((jacobian ** 2).sum()).item())

jacobian shape: (576, 576)
jacobian norm: 32.25


In [ ]:
# Same thing but to do it for all positions in the sequence, not just pos, so we get a 3D Jacobian tensor of shape [d_model_j, seq_len, d_model_i] which we can mask and average as needed for J-Lens

d_model_j = h_j.shape[-1]

basis = mx.eye(d_model_j)
cotangents = mx.zeros((d_model_j,) + h_j.shape)
basis_enlarged = rearrange(basis, "d_model_j d_model_i -> d_model_j 1 1 d_model_i")  # [d_model_j, 1, 1, d_model_i]
print("basis_enlarged shape:", basis_enlarged.shape)  # [d_model_j, seq_len, d_model_j]
cotangents = mx.repeat(basis_enlarged, cotangents.shape[2], axis=2)  # repeat along seq_len dimension

print("cotangents shape:", cotangents.shape)  # [d_model_j, 1, seq_len, d_model_i]
for pos in range(min(3, h_j.shape[1])):
    print(f"cotangents[:, 0, {pos}, :]:\n", cotangents[:, 0, pos, :])

basis_enlarged shape: (576, 1, 1, 576)
cotangents shape: (576, 1, 8, 576)
cotangents[:, 0, 0, :]:
 array([[1, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 0, 0, 1]], dtype=float32)
cotangents[:, 0, 1, :]:
 array([[1, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 0, 0, 1]], dtype=float32)
cotangents[:, 0, 2, :]:
 array([[1, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 0, 0, 1]], dtype=float32)


In [ ]:
# vjp_row function is the same as before, but now we can compute the Jacobian for all positions in the sequence
jac_rows = mx.vmap(vjp_row)(cotangents)  # [d_model_j, 1, seq_len, d_model_i]
print("jac_rows shape:", jac_rows.shape)  # [d_model_j, 1, seq_len, d_model_i]
jacobian = jac_rows[:, 0, :, :]  # [d_model_j, seq_len, d_model_i]
print("jacobian shape:", jacobian.shape)  # [d_model_j, seq_len, d_model_i]
print("jacobian norm:", mx.sqrt((jacobian ** 2).sum()).item())

jac_rows shape: (576, 1, 8, 576)
jacobian shape: (576, 8, 576)
jacobian norm: 127.5


In [46]:
for pos in range(min(3, h_j.shape[1])):
    print(f"jacobian[:, {pos}, :]:\n", jacobian[:, pos, :])

jacobian[:, 0, :]:
 array([[1.0625, -0.00482178, -0.00367737, ..., 0.0195312, 0.112793, 0.0180664],
       [-0.00285339, 0.494141, -0.0390625, ..., -0.119629, -0.140625, -0.0125732],
       [0.0139771, -0.0200195, 1.01562, ..., -0.0703125, -0.0529785, -0.0339355],
       ...,
       [0.0110474, -0.0529785, 0.0281982, ..., 0.34375, 0.0458984, 0.140625],
       [0.0444336, 0.0361328, 0.0859375, ..., 0.074707, 0.582031, 0.0410156],
       [0.00100708, -0.0644531, -0.0303955, ..., -0.059082, -0.0288086, 0.462891]], dtype=bfloat16)
jacobian[:, 1, :]:
 array([[1.32812, 0.0239258, -0.141602, ..., -0.0050354, -0.0142822, -0.0620117],
       [-0.0732422, 0.207031, -0.0722656, ..., 0.0422363, 0.0825195, -0.0429688],
       [-0.0712891, -0.0698242, 1.03906, ..., 0.0947266, -0.149414, -0.125],
       ...,
       [-0.0664062, -0.0849609, 0.026123, ..., 0.3125, 0.0314941, 0.0205078],
       [-0.00354004, 0.144531, 0.0444336, ..., 0.0605469, 0.337891, 0.0169678],
       [0.0106201, -0.0996094, -0.131

## 6. Recap: PyTorch/TransformerLens vs MLX

| Capability | PyTorch + TransformerLens | MLX |
|---|---|---|
| Device | explicit `.to("mps")`, correctness caveat | unified memory, GPU by default, no caveat |
| Read an activation | `run_with_cache` / `HookPoint` | call the submodules yourself |
| Patch an activation | `run_with_hooks(fwd_hooks=...)` | call the submodules yourself, substitute a value |
| Parameter gradient | `loss.backward()`, read `.grad` | `nn.value_and_grad(model, loss_fn)`, returns a pytree |
| Activation gradient | `bwd_hooks` (module-level, order-sensitive) | `mx.vjp(fn, primals, cotangents)`, a plain function call |
| Execution | eager | lazy — call `mx.eval(...)` (or `.item()`) to materialize |